In [4]:
import pandas as pd
import numpy as np

file_name = 'paket_a_transaksi_pos_2024_2025.csv'
df = pd.read_csv(file_name)

print("Total data awal:", len(df))
display(df.head())

print("\nMissing value awal:")
print(df.isnull().sum())

df_clean = df.copy()

# handle missing val
df_clean['kode_kasir'] = df_clean['kode_kasir'].fillna('Unknown')

# konversi kolom numerik biar pasti beritip angka (mencegah error TypeError)
num_cols = ['jumlah', 'harga_satuan', 'diskon_persen']
for col in num_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(0)

# filter status
status_valid = ['SELESAI', 'Selesai', 'SUCCESS', 'Success', 'Completed', 'Paid']
df_valid = df_clean[df_clean['status'].astype(str).str.strip().isin(status_valid)].copy()

# get year dari tanggal
df_valid['tanggal_dt'] = pd.to_datetime(df_valid['tanggal'], format='mixed', dayfirst=True)
df_valid['tahun'] = df_valid['tanggal_dt'].dt.year

# hitung pendapatan kotor
subtotal = df_valid['harga_satuan'] * df_valid['jumlah']
diskon_nominal = subtotal * (df_valid['diskon_persen'] / 100)
df_valid['pendapatan'] = subtotal - diskon_nominal

# split per year
df_2024 = df_valid[df_valid['tahun'] == 2024].copy()
df_2025 = df_valid[df_valid['tahun'] == 2025].copy()

total_2024 = df_2024['pendapatan'].sum()
total_2025 = df_2025['pendapatan'].sum()

print("\n--- Summary ---")
print(f"2024 ({len(df_2024)} rows): Rp {total_2024:,.2f}")
print(f"2025 ({len(df_2025)} rows): Rp {total_2025:,.2f}")

# export output
df_summary = pd.DataFrame({
    'Tahun': [2024, 2025],
    'Jumlah_Data': [len(df_2024), len(df_2025)],
    'Total_Pendapatan': [total_2024, total_2025]
})
df_summary.to_csv('ringkasan_pendapatan_2024_2025.csv', index=False)
df_valid.to_csv('dataset_cleaned.csv', index=False)

Total data awal: 5030


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,"7500,00",0,SELESAI
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500,0,SELESAI
4,TRX0243509,26/03/2025,G05,K016,RMT1407,1,6000,0,SELESAI



Missing value awal:
id_transaksi       0
tanggal            0
kode_gerai         0
kode_kasir       261
kode_produk        0
jumlah             0
harga_satuan       0
diskon_persen      0
status             0
dtype: int64

--- Summary ---
2024 (2364 rows): Rp 385,920,077.40
2025 (2385 rows): Rp 244,899,662.90
